# 00 Sudan demo data

Inspects the U.S. Census Bureau Sudan files from HDX and writes `data/processed/sudan_states.gpkg`.

Before running: download `Sudan.gdb.zip` and `Sudan_uscb_202101.xlsx` from https://data.humdata.org/dataset/sudan-subnational-boundaries-and-tabular-data into `data/raw/`, then unzip `Sudan.gdb.zip` there so you have `data/raw/Sudan.gdb/`.

In [3]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Notebook runs from notebooks/, so data sits one level up
RAW = Path("../data/raw")
OUT = Path("../data/processed")
# The real geodatabase is the folder holding .gdbtable files, however deep the unzip put it
GDB = next(p.parent for p in RAW.rglob("*.gdbtable"))
XLSX = next(RAW.glob("*.xlsx"))
print(GDB, XLSX)

..\data\raw\sudan.gdb\Sudan.gdb ..\data\raw\sudan_uscb_202101.xlsx


In [4]:
# List every layer in the geodatabase with its geometry type
gpd.list_layers(GDB)

,name,geometry_type
0,SD_GEOG_ADM0_uscb_202101,MultiPolygon
1,SD_GEOG_ADM1_2018Geog_uscb_202101,MultiPolygon
2,SD_GEOG_ADM1_CensusGeog_2008_uscb_202101,MultiPolygon
3,SD_GEOG_ADM1_RetabGeog_uscb_202101,MultiPolygon
4,SD_GEOG_ADM2_2018Geog_uscb_202101,MultiPolygon
5,SD_GEOG_ADM2_RetabGeog_uscb_202101,MultiPolygon
6,SD_GEOG_ADM3_RetabGeog_uscb_202101,MultiPolygon
7,SD_AMENITY_2018Geog_survey2014_uscb_202101,NaN
8,SD_CHILD_WELL_BEING_2018Geog_survey2014_uscb_2...,NaN
9,SD_ECONOMY_CensusGeog_census2008_uscb_202101,NaN


Pick the state-level (admin 1) layer from the list above and set its name below.

In [14]:
# Set this to the admin 1 layer name shown above
STATE_LAYER = "SD_GEOG_ADM1_RetabGeog_uscb_202101"

states_raw = gpd.read_file(GDB, layer=STATE_LAYER)
print(states_raw.crs, len(states_raw))
states_raw.drop(columns="geometry").head()

EPSG:4326 18


,AREA_NAME,GEO_CONCAT,GEO_MATCH,CNTRY_NAME,ADM1_NAME,ADM_LEVEL,USCBCMNT,GENC_CODE,FIPS_CODE,NSO_CODE,NSO_NAME,Shape_Length,Shape_Area
0,BLUE NILE,SUDAN_BLUE NILE,SDN_01,SUDAN,BLUE NILE,1,An Nīl al Azraq.,SD-NB,SU42,44,BLUE NILE (ولايـــة النيل الأزرق ),8.933760,3.137437
1,CENTRAL DARFUR,SUDAN_CENTRAL DARFUR,SDN_02,SUDAN,CENTRAL DARFUR,1,Wasaţ Dārfūr. Split from West Darfur.,SD-DC,SU61,,CENTRAL DARFUR,8.930015,2.586913
2,EAST DARFUR,SUDAN_EAST DARFUR,SDN_03,SUDAN,EAST DARFUR,1,Eastern Darfur; Sharq Dārfūr. Split from Sout...,SD-DE,SU60,,EAST DARFUR,10.961586,4.382658
3,GEDAREF,SUDAN_GEDAREF,SDN_04,SUDAN,GEDAREF,1,Al-Gadarif; Al Gedaref; Al Qaḑārif; Qadarif.,SD-GD,SU39,23,ALGADARIF (ولاية القضارف),10.918341,4.983354
4,GEZIRA,SUDAN_GEZIRA,SDN_05,SUDAN,GEZIRA,1,Aj Jazira; Aj Jazirah; Jazirah; Al Gazira; Al ...,SD-GZ,SU38,41,ALGAZEERA (ولاية الجزيرة),8.779387,2.280391


In [15]:
# List the sheets in the population workbook
pd.ExcelFile(XLSX).sheet_names

['Metadata',
 'Data Dictionary',
 'Population_CensusGeog',
 'Population_RetabGeog',
 'Population Estimates_2020Geog',
 'Child Well-Being_2018Geog',
 'Economy_CensusGeog',
 'Nationality_CensusGeog',
 'Migration_2018Geog',
 'Household_CensusGeog',
 'Household_2018Geog',
 'Housing Unit_2018Geog',
 'Amenity_2018Geog']

In [16]:
# Set this to the sheet with population by state, then check where the header row is
SHEET = "Population_RetabGeog"

pop_raw = pd.read_excel(XLSX, sheet_name=SHEET, header=None)
pop_raw.head(15)

c:\Users\dev\Desktop\Projects\kharita\.venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,0,1,2,3,4,5,6,7,8,9,...,42,43,44,45,46,47,48,49,50,51
0,AREA_NAME,GEO_CONCAT,GEO_MATCH,CNTRY_NAME,ADM1_NAME,ADM2_NAME,ADM3_NAME,ADM_LEVEL,USCBCMNT,GENC_CODE,...,POP_UFM_I,POP_RUR_I,POP_RML_I,POP_RFM_I,POP_NOM_I,POP_NML_I,POP_NFM_I,POP_HH_T,POP_HH_N,POP_HH_I
1,Country or subnational area name,Administrative name concatenation,Geographic match,Country name,First-order administrative division name,Second-order administrative division name,Third-order administrative division name,Administrative level,Notes,GENC code,...,"Urban Female Population, institutional population","Rural Population, institutional population","Rural Male Population, institutional population","Rural Female Population, institutional population","Nomadic Population, institutional population","Nomadic Male Population, institutional population","Nomadic Female Population, institutional popul...","Households, institutional and non-institutiona...","Households, non-institutional population","Households, institutional population"
2,SUDAN,SUDAN,SDN_00,SUDAN,NaN,NaN,NaN,0,Republic of the Sudan; As Sūdān; Jumhūrīyat as...,SD,...,41288,136958,126784,10174,5494,4232,1262,5390135,5369371,20770
3,BLUE NILE,SUDAN_BLUE NILE,SDN_01,SUDAN,BLUE NILE,NaN,NaN,1,An Nīl al Azraq.,SD-NB,...,407,9325,8602,723,6,3,3,145723,145188,535
4,AL TADAMON,SUDAN_BLUE NILE_AL TADAMON,SDN_01_01,SUDAN,BLUE NILE,AL TADAMON,NaN,2,NaN,NaN,...,0,247,210,37,0,0,0,13796,13765,31
5,BUT,SUDAN_BLUE NILE_AL TADAMON_BUT,SDN_01_01_01,SUDAN,BLUE NILE,AL TADAMON,BUT,3,Booat.,NaN,...,0,153,140,13,0,0,0,7797,7781,16
6,RŪRU,SUDAN_BLUE NILE_AL TADAMON_RŪRU,SDN_01_01_02,SUDAN,BLUE NILE,AL TADAMON,RŪRU,3,Roro.,NaN,...,0,94,70,24,0,0,0,5999,5984,15
7,AR RUSERIS,SUDAN_BLUE NILE_AR RUSERIS,SDN_01_02,SUDAN,BLUE NILE,AR RUSERIS,NaN,2,NaN,NaN,...,41,3330,3203,127,0,0,0,35696,35579,117
8,AR RUSERIS,SUDAN_BLUE NILE_AR RUSERIS_AR RUSERIS,SDN_01_02_01,SUDAN,BLUE NILE,AR RUSERIS,AR RUSERIS,3,Al Roseires; Alrosiris Town and Rural.,NaN,...,41,143,107,36,0,0,0,12366,12342,24
9,BADOS,SUDAN_BLUE NILE_AR RUSERIS_BADOS,SDN_01_02_02,SUDAN,BLUE NILE,AR RUSERIS,BADOS,3,Badous; Badose.,NaN,...,0,808,774,34,0,0,0,10435,10386,49


Set the header row, the join key in each table, and the population column, using what you saw above.

In [17]:
HEADER_ROW = 0
KEY_IN_STATES = "GEO_MATCH"
KEY_IN_POP = "GEO_MATCH"

# Row 1 holds descriptions, so read names and descriptions together
names = pop_raw.iloc[0]
descs = pop_raw.iloc[1].astype(str)
# Total population is the column described as total, institutional and non-institutional
match = names[descs.str.startswith("Total Population, institutional and non-institutional")]
print(match.tolist())
POP_COLUMN = match.iloc[0]

pop = pd.read_excel(XLSX, sheet_name=SHEET, header=HEADER_ROW, skiprows=[1])
# Keep state rows only
pop = pop[pd.to_numeric(pop["ADM_LEVEL"], errors="coerce") == 1]
pop = pop[[KEY_IN_POP, "AREA_NAME", POP_COLUMN]].rename(
    columns={KEY_IN_POP: "key", "AREA_NAME": "pop_name", POP_COLUMN: "pop"}
)
pop["pop"] = pd.to_numeric(pop["pop"], errors="coerce")
print(POP_COLUMN, len(pop), "states")
pop.head()

['POP_TP_T']
POP_TP_T 18 states


c:\Users\dev\Desktop\Projects\kharita\.venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,key,pop_name,pop
1,SDN_01,BLUE NILE,832112.0
27,SDN_02,CENTRAL DARFUR,553515.0
52,SDN_03,EAST DARFUR,1216719.0
76,SDN_04,GEDAREF,1348378.0
107,SDN_05,GEZIRA,3575280.0


In [18]:
# Join, report anything that failed to match
states = states_raw.rename(columns={KEY_IN_STATES: "key"})
states = states.merge(pop, on="key", how="left", indicator=True)
print(states["_merge"].value_counts())
states[states["_merge"] != "both"][["key"]]

_merge
both          18
left_only      0
right_only     0
Name: count, dtype: int64


,key


In [19]:
# Every row should say True; any False means the codes point at different states
check = states[["key", "AREA_NAME", "pop_name"]].copy()
check["same"] = check["AREA_NAME"].str.upper() == check["pop_name"].str.upper()
check

,key,AREA_NAME,pop_name,same
0,SDN_01,BLUE NILE,BLUE NILE,True
1,SDN_02,CENTRAL DARFUR,CENTRAL DARFUR,True
2,SDN_03,EAST DARFUR,EAST DARFUR,True
3,SDN_04,GEDAREF,GEDAREF,True
4,SDN_05,GEZIRA,GEZIRA,True
5,SDN_06,KASSALA,KASSALA,True
6,SDN_07,KHARTOUM,KHARTOUM,True
7,SDN_08,NORTH DARFUR,NORTH DARFUR,True
8,SDN_09,NORTH KORDOFAN,NORTH KORDOFAN,True
9,SDN_10,NORTHERN,NORTHERN,True


In [20]:
# Area in km2 from an equal-area projection, then density
states["area_km2"] = states.to_crs("ESRI:102022").area / 1e6
states["density"] = states["pop"] / states["area_km2"]
states[["key", "pop", "area_km2", "density"]].sort_values("density")

,key,pop,area_km2,density
9,SDN_10,699065.0,364240.005893,1.919243
10,SDN_11,1396110.0,215644.767167,6.474120
7,SDN_08,2113626.0,316228.589588,6.683855
11,SDN_12,1120441.0,130387.888467,8.593137
8,SDN_09,2039495.0,187124.170147,10.899153
14,SDN_15,867918.0,78491.660161,11.057455
16,SDN_17,1419983.0,113969.568163,12.459317
1,SDN_02,553515.0,31116.294817,17.788590
0,SDN_01,832112.0,37887.377919,21.962776
3,SDN_04,1348378.0,59520.548647,22.653991


In [21]:
# Keep a small clean file in EPSG:4326, simplified to keep the demo light
keep = states[["key", "pop", "area_km2", "density", "geometry"]].to_crs("EPSG:4326")
keep["geometry"] = keep.geometry.simplify(0.005, preserve_topology=True)
OUT.mkdir(parents=True, exist_ok=True)
keep.to_file(OUT / "sudan_states.gpkg", driver="GPKG")
print((OUT / "sudan_states.gpkg").stat().st_size / 1e6, "MB")

0.159744 MB


In [22]:
# Quick visual check with plain Lonboard
from lonboard import viz

viz(keep)